In [714]:
import os
import json
import time
import requests
import numpy as np
import twelvelabs
from dotenv import load_dotenv
from twelvelabs.models.embed import SegmentEmbedding
from typing import List
import requests
import io
from PIL import Image, ImageOps
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import pickle
load_dotenv()

True

In [715]:
with open("/home/ubuntu/code/libby/pipeline/data/finetuned_may19.pkl", 'rb') as file:
    resnet_df = pickle.load(file)
    
resnet_df = resnet_df[['video', 'frame', 'owl_label', 'finetuned_embedding']]
resnet_df

/tmp/ipykernel_340466/3378888104.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  resnet_df = pickle.load(file)


,video,frame,owl_label,finetuned_embedding
0,Scenes 001-020__314-3_20230815232058756.mp4,112,Beige collared shirt,"[0.0048876973, 0.019972654, 0.048866335, 0.001..."
1,Scenes 001-020__314-3_20230815232058756.mp4,58,Beige collared shirt,"[0.0041305386, 0.026740564, 0.038905628, 0.001..."
2,Scenes 001-020__314-3_20230815232058756.mp4,115,Beige collared shirt,"[0.00619403, 0.01454955, 0.044577274, 0.003630..."
3,Scenes 001-020__314-3_20230815232058756.mp4,83,Beige collared shirt,"[0.009226867, 0.020863937, 0.03188111, 0.00792..."
4,Scenes 001-020__314-3_20230815232058756.mp4,52,Dark coat,"[0.014232857, 0.037429027, 0.03028806, 0.00619..."
...,...,...,...,...
18323,Scenes 001-020__303L-1-_20230815223024888.mp4,83,Horned helmet,"[0.0034361978, 0.01590296, 0.03698839, 0.01353..."
18324,Scenes 001-020__303L-1-_20230815223024888.mp4,98,Dark coat,"[0.0027844398, 0.0069978, 0.03899962, 0.002111..."
18325,Scenes 001-020__303L-1-_20230815223024888.mp4,6,Machete,"[0.00011212564, 0.037287023, 0.050194696, 0.02..."
18326,Scenes 001-020__303L-1-_20230815223024888.mp4,40,Brown jacket,"[0.0036044263, 0.029153472, 0.048092548, 0.022..."


In [716]:
### DREW EDIT THIS LINE
# with open("/home/ubuntu/code/libby/pipeline/data/definitiveObjects_jeremiah.pkl", 'rb') as file:
#     defObjects = pickle.load(file)

# class_embs = (
#     defObjects
#     .groupby('class')
#     .agg({
#         'finetuned_embedding': lambda x: np.mean(np.vstack(x), axis=0),
#         # 'text_embedding': lambda x: np.mean(np.vstack(x), axis=0)
#     })
#     .reset_index()
# )



In [717]:
with open("/home/ubuntu/code/contrastive_results/contrastive_embeddings_viz_e75_t0.7_dim2048/prototypes.pkl", 'rb') as file:
    defObjects = pickle.load(file)

class_emb_matrix = list(defObjects['representative_embedding'])
objs = list(defObjects['class'])

In [718]:
def cosine_scores(input_emb, class_emb_matrix):
    input_tensor = F.normalize(torch.tensor(input_emb, dtype=torch.float32).unsqueeze(0), dim=1)
    class_tensor = F.normalize(torch.tensor(np.vstack(class_emb_matrix), dtype=torch.float32), dim=1)
    return F.cosine_similarity(input_tensor, class_tensor).numpy()



def visual_prediction_from_text_filtered(row):
    visual_scores = cosine_scores(row['finetuned_embedding'], class_emb_matrix)
    best_idx = np.argmax(visual_scores)
    return pd.Series({
        'visual_predicted_object': objs[best_idx],
        'visual_max_score': visual_scores[best_idx],
    })

visual_preds = resnet_df.apply(visual_prediction_from_text_filtered, axis=1)
resnet_df[['visual_predicted_object', 'visual_max_score']] = visual_preds

In [719]:
resnet_df

,video,frame,owl_label,finetuned_embedding,visual_predicted_object,visual_max_score
0,Scenes 001-020__314-3_20230815232058756.mp4,112,Beige collared shirt,"[0.0048876973, 0.019972654, 0.048866335, 0.001...",Sylvie's Armor,0.487312
1,Scenes 001-020__314-3_20230815232058756.mp4,58,Beige collared shirt,"[0.0041305386, 0.026740564, 0.038905628, 0.001...",Sylvie's Armor,0.505095
2,Scenes 001-020__314-3_20230815232058756.mp4,115,Beige collared shirt,"[0.00619403, 0.01454955, 0.044577274, 0.003630...",Sylvie's Armor,0.477848
3,Scenes 001-020__314-3_20230815232058756.mp4,83,Beige collared shirt,"[0.009226867, 0.020863937, 0.03188111, 0.00792...",Sylvie's Armor,0.500138
4,Scenes 001-020__314-3_20230815232058756.mp4,52,Dark coat,"[0.014232857, 0.037429027, 0.03028806, 0.00619...",Sylvie's Armor,0.569488
...,...,...,...,...,...,...
18323,Scenes 001-020__303L-1-_20230815223024888.mp4,83,Horned helmet,"[0.0034361978, 0.01590296, 0.03698839, 0.01353...",Sylvie's Armor,0.481301
18324,Scenes 001-020__303L-1-_20230815223024888.mp4,98,Dark coat,"[0.0027844398, 0.0069978, 0.03899962, 0.002111...",Sylvie's Armor,0.471336
18325,Scenes 001-020__303L-1-_20230815223024888.mp4,6,Machete,"[0.00011212564, 0.037287023, 0.050194696, 0.02...",Sylvie's Armor,0.449510
18326,Scenes 001-020__303L-1-_20230815223024888.mp4,40,Brown jacket,"[0.0036044263, 0.029153472, 0.048092548, 0.022...",Sylvie's Armor,0.493889


In [ ]:
visual_threshold = 0.45 # THIS WILL CHANGE -- NEED TO TEST FOR YOUR STUFF
resnet_df['prediction'] = 1
resnet_df['frame'] = resnet_df['frame'].apply(lambda row: int(row))

# idx = filtered_df_copy.groupby(['video', 'visual_predicted_object'])['prediction'].idxmax()
idx = resnet_df.groupby(['video', 'visual_predicted_object'])['visual_max_score'].idxmax()
resnet_df = resnet_df.loc[idx].reset_index(drop=True)

resnet_df['visual_predicted_object'] = resnet_df.apply(lambda row: row['visual_predicted_object'] if row['visual_max_score'] > visual_threshold else 'No Class', axis=1)
resnet_df = resnet_df[resnet_df['visual_predicted_object'] != 'No Class']


In [721]:
with open("/home/ubuntu/code/libby/pipeline/data/sourceTruth_jeremiah.pkl", 'rb') as file:
    final_SOT = pickle.load(file)
    
final_SOT = final_SOT.rename(columns={'frame': 'second', 'second': 'frame'})
final_SOT = final_SOT.groupby(['video', 'tag'])['actual'].max().reset_index()
# final_SOT['frame'] = final_SOT['frame'].apply(lambda row: int(row))

merged = pd.merge(resnet_df, final_SOT, left_on=['video', 'visual_predicted_object'], right_on=['video', 'tag'], how='right')

/tmp/ipykernel_340466/3172387393.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  final_SOT = pickle.load(file)


In [722]:
merged

,video,frame,owl_label,finetuned_embedding,visual_predicted_object,visual_max_score,prediction,tag,actual
0,Scenes 001-020__101B-1-_20230726152900590.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Aligator Loki Plushie,0.0
1,Scenes 001-020__101B-1-_20230726152900590.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Boastful Loki Armor,0.0
2,Scenes 001-020__101B-1-_20230726152900590.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Classic Loki Armor,0.0
3,Scenes 001-020__101B-1-_20230726152900590.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Kid Loki Armor,0.0
4,Scenes 001-020__101B-1-_20230726152900590.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Loki's Armor,1.0
...,...,...,...,...,...,...,...,...,...
1813,Scenes 061-080__265N-1_20230815220141511.mp4,NaN,NaN,NaN,NaN,NaN,NaN,TVA Prisoner Uniform,0.0
1814,Scenes 061-080__265N-1_20230815220141511.mp4,NaN,NaN,NaN,NaN,NaN,NaN,TVA Uniform,0.0
1815,Scenes 061-080__265N-1_20230815220141511.mp4,NaN,NaN,NaN,NaN,NaN,NaN,TemPad,0.0
1816,Scenes 061-080__265N-1_20230815220141511.mp4,NaN,NaN,NaN,NaN,NaN,NaN,Time Stick,1.0


In [723]:
def classify_answer(answer, pred):
    if answer == 0 and pred == 0:
        return 'TN'
    elif answer == 1 and pred == 1:
        return 'TP'
    elif answer == 1 and pred == 0:
        return 'FN'
    return 'FP'
merged['prediction'] = merged['prediction'].fillna(0)
merged['answerClass'] =  merged.apply(lambda row: classify_answer(row['actual'], row['prediction']), axis = 1)

In [724]:
counts = merged['answerClass'].value_counts()
TP = counts.get('TP', 0)
FP = counts.get('FP', 0)
FN = counts.get('FN', 0)
TN = counts.get('TN', 0)

# Calculate metrics
print('TP: ', TP)
print('FP: ', FP)
print('FN: ', FN)
print('TN: ', TN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# Display
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1_score:.4f}')

TP:  27
FP:  128
FN:  109
TN:  1554
Precision: 0.1742
Recall:    0.1985
F1 Score:  0.1856


In [725]:
alx = merged.groupby(["tag","answerClass"]).size().reset_index().pivot(columns='answerClass', values=0, index = 'tag').fillna(0).reset_index().sort_values('TP')
alx['precision'] = alx['TP'] / (alx['TP'] + alx['FP']) 
alx['recall'] = alx['TP'] / (alx['TP'] + alx['FN']) 
alx['f-score'] = (alx['precision']*alx['recall']/(alx['precision'] + alx['recall']))*2
 
alx['Total Labels Across Videos'] = alx['TP'] + alx['FN']
 
alx.fillna(0).sort_values('f-score', ascending = False)

answerClass,tag,FN,FP,TN,TP,precision,recall,f-score,Total Labels Across Videos
8,Sylvie's Armor,0.0,78.0,6.0,17.0,0.178947,1.000000,0.303571,17.0
5,Loki's Variant uniform,15.0,9.0,72.0,5.0,0.357143,0.250000,0.294118,20.0
10,Sylvie's machete,2.0,41.0,53.0,5.0,0.108696,0.714286,0.188679,7.0
0,Aligator Loki Plushie,3.0,0.0,98.0,0.0,0.000000,0.000000,0.000000,3.0
1,Boastful Loki Armor,3.0,0.0,98.0,0.0,0.000000,0.000000,0.000000,3.0
2,Classic Loki Armor,4.0,0.0,97.0,0.0,0.000000,0.000000,0.000000,4.0
7,Ravonna Renslayer's Uniform,4.0,0.0,97.0,0.0,0.000000,0.000000,0.000000,4.0
3,Kid Loki Armor,3.0,0.0,98.0,0.0,0.000000,0.000000,0.000000,3.0
4,Loki's Armor,4.0,0.0,97.0,0.0,0.000000,0.000000,0.000000,4.0
6,Loki's dagger,1.0,0.0,100.0,0.0,0.000000,0.000000,0.000000,1.0
